# ConvNeXt-Tiny + FPN Nowcasting v2

Swin v2と同じ6バンド対応、衛星別正規化・Stem、時間特徴、location非重複5-foldを使用します。公式ConvNeXt実装と公開重みはMITライセンスです。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q timm rasterio
!nvidia-smi | head -5

## パス設定

TIFFはDriveから直接読むと極端に遅いため、データは`/content`へ展開し、モデルだけDriveへ保存します。

In [ ]:
from pathlib import Path
import sys

ROOT = Path('/content')
DRIVE_ROOT = Path('/content/drive/MyDrive/solafune-workspace')
sys.path.insert(0, str(DRIVE_ROOT / 'src'))

for required in ('train_dataset', 'evaluation_dataset', 'sample_submission'):
    assert (ROOT / required).is_dir(), f'Not found: {ROOT / required}'

drive_models = DRIVE_ROOT / 'models'
drive_models.mkdir(parents=True, exist_ok=True)
local_models = ROOT / 'models'
if local_models.is_symlink() and local_models.resolve() != drive_models.resolve():
    local_models.unlink()
if not local_models.exists():
    local_models.symlink_to(drive_models, target_is_directory=True)

In [ ]:
from swin_nowcast_v2 import Config, create_submission, get_device, make_folds, prepare_metadata
from convnext_nowcast_v2 import ensemble_convnext_predict, train_convnext_fold

device = get_device()
config = Config(
    root=str(ROOT),
    encoder_size=224,
    batch_size=8,          # T4でOOMなら4
    epochs=15,
    lr_encoder=5e-5,
    lr_head=2e-4,
    workers=4,
    pretrained=True,
    use_amp=True,
    band_mode='matched6',
    use_satellite_normalization=True,
    convnext_model_subdir='convnext_v2',
)
train_df = prepare_metadata(config.train_dir / 'train_dataset.csv')
eval_df = prepare_metadata(config.evaluation_dir / 'evaluation_target.csv')
folds = make_folds(train_df, config.n_folds)
print(device, len(train_df), len(eval_df))

## Fold 0検証

まずfold 0だけ学習します。目標はSwinの`1.2943`、最低ラインはU-Netの`1.3170`です。

In [ ]:
TRAIN_FOLDS = [0]
results = []
for fold_index in TRAIN_FOLDS:
    result = train_convnext_fold(config, train_df, folds[fold_index], device=device)
    results.append(result)
print({r['fold']: r['validation_rmse'] for r in results})

## 5-fold本学習

Fold 0が有望な場合のみ実行します。既存チェックポイントの自動スキップはしないため、完了済みfoldをリストから外してください。

In [ ]:
TRAIN_FOLDS = [1, 2, 3, 4]
results = []
for fold_index in TRAIN_FOLDS:
    result = train_convnext_fold(config, train_df, folds[fold_index], device=device)
    results.append(result)
print({r['fold']: r['validation_rmse'] for r in results})

## 推論と提出ZIP

In [ ]:
predictions = ensemble_convnext_predict(config, eval_df, list(range(5)), device=device)
print(predictions.shape, predictions.min(), predictions.mean(), predictions.max())
output_zip = DRIVE_ROOT / 'submission_convnext_v2.zip'
create_submission(config, eval_df, predictions, output_zip)
print(output_zip)